# 歌词分词，词性标注

In [2]:
import json
import pandas as pd


# import thulac


from collections import Counter
from openai import OpenAI

In [3]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [4]:
import sys, os
sys.path.append('..')

# 分词，词频与词性分析

In [5]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [6]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [7]:
import re
from hanlp_restful import HanLPClient


api = "https://hanlp.com/hanlp/v21/redirect"
# api = "https://hanlp.hankcs.com/api"
# api = "https://www.hanlp.com/api"
HanLP = HanLPClient(api, auth="699691e7eaf61a3aca90d7b8", language='zh')


def is_chinese_word(word):
    """
    判断是否为纯中文词
    """
    return 1 if re.fullmatch(r'[\u4e00-\u9fff]+', word) else 0

def is_english_word(word):
    """
    判断是否为纯英文词
    """
    return 1 if re.fullmatch(r'[a-zA-Z]+', word) else 0



def process_lyrics_with_hanlp_multi_pos(text, word_to_fix=None):
    if not text:
        return []

    # 调用 HanLP
    result = HanLP.parse(text, tasks='pos/pku')

    sentences = result['tok/fine']
    pos_sentences = result['pos/pku']

    # 统计 (word, pos) -> freq
    word_pos_counter = Counter()

    for words, pos_tags in zip(sentences, pos_sentences):
        for word, tag in zip(words, pos_tags):

            word = word.strip()

            # 过滤标点
            if tag == 'w' or not word:
                continue

            # 词性修正
            if word_to_fix and word in word_to_fix:
                tag = word_to_fix[word]

            word_pos_counter[(word, tag)] += 1

    # 构建结果列表
    results = []
    for (word, pos), freq in word_pos_counter.items():
        results.append({
            "word": word,
            "pos": pos,
            "freq": freq,
            "is_chinese": is_chinese_word(word)
        })

    # 按词频排序
    results.sort(key=lambda x: x["freq"], reverse=True)

    return results


In [8]:
# thu = thulac.thulac(seg_only=False, filt=True) 

# def process_lyrics_with_thulac(text, word_to_fix=None):
#     if not text:
#         return []
    
#     # 2. 执行分词与词性标注
#     # 返回格式为 [[word, pos], [word, pos], ...]
#     words_with_pos = thu.cut(text)
    
#     # 3. 过滤无意义字符与词性修正
#     # thulac 的标点词性通常是 'w'
#     filtered_data = []
#     for word, pos in words_with_pos:
#         word = word.strip()
#         # 排除标点符号、空白字符
#         if pos != 'w' and len(word) > 0:
#             # 逻辑修正：word_to_fix 通常是修正词性
#             if word_to_fix and word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 4. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 5. 汇总信息
#     # 建立 word -> pos 映射
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count
#         })
    
#     return sorted_results

In [9]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'

    # 已解析的内容
    songs_id_exist = []
    if os.path.exists(path_prefix + "raw_words_data.csv"):
        df_exist = pd.read_csv(path_prefix + "raw_words_data.csv")
        songs_id_exist = df_exist['song_id'].unique().tolist()

    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i and i['song_id'] not in songs_id_exist:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            # lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
            #     i['lyrics_text'], word_to_fix=word_to_fix)
            print(i['song_name'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_hanlp_multi_pos(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    # 合并df_exis和 df_word
    if 'df_exist' in locals():
        df_word = pd.concat([df_exist, df_word], ignore_index=True)
    return df_word

In [27]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word = df_word.copy()
    df_songs = df_songs.copy()
    df_word['song_id'] = df_word['song_id'].astype(str)
    df_word['word'] = df_word['word'].astype(str)
    df_word['is_chinese'] = df_word['word'].apply(is_chinese_word)
    df_word['is_english'] = df_word['word'].apply(is_english_word)
    # 把英文词转为小写
    df_word.loc[df_word['is_english'] == 1,
                'word'] = df_word.loc[df_word['is_english'] == 1,
                                      'word'].str.lower()
    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    # df_merged = df_word.merge(df_songs, on='song_id', how='left')
    df_merged = df_songs.merge(df_word, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# 批量采集

In [ ]:
singers = [('luodayou', '罗大佑'), ('lizongsheng', '李宗盛'), ('zhangxueyou', '张学友'), ('twins', 'Twins'), ('wangsulong', '汪苏泷'), ('panweibo', '潘玮柏'), ('dengziqi', 'G.E.M. 邓紫棋'), ('xuezhiqian', '薛之谦'), ('xusong', '许嵩'), ('zhangjie', '张杰'), ('taozhe', '陶喆'), ('fangdatong', '方大同'), ('wangfei', '王菲'), ('maobuyi', '毛不易'), ('beyond', 'BEYOND')]
for i in singers[-1:]:
    file_path_prefix = f'data/{i[0]}/'
    # 歌曲数据
    df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
    # 词性解析
    # hanlp暂时不需要word_to_fix
    df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
    df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
    # 重新读取
    df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")
    df_merged = words_data_merge(df_word_read, df_songs)
    df_merged = df_merged.dropna(subset='song_name')
    # 过滤中文词
    df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
    # 虚拟专辑数据
    df_songs_part = df_merged_chn[[
        'song_name_pure'
    ]].drop_duplicates(keep='first').reset_index(drop=True)
    # 只保留120个
    df_songs_part = df_songs_part.head(100)
    df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                            1).astype(str)
    df_songs_part['album_order'] = df_songs_part.index // 10
    # 虚拟专辑数据，index//12+1作为虚拟专辑
    df_merged_chn = df_merged_chn.copy()
    df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
    df_merged_chn = df_merged_chn.drop(columns=['album_name'])
    df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
    # 删除album_order为空的数据
    df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
    df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)
    df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)
    df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# main

In [11]:
singer_list = [
        'mayday', 'jaychou', 'liyuchun', 'chenyixun', 'renxianqi', 'linjunjie',
        'sunyanzi', 'remen', 'fangwenshan', 'chenxinhong', 'caiyilin', 'wubai', 'zhoushen', 'zhoushen_pure', 'fenghuangchuanqi', 'wanglihong', 'liangjingru', 'wangxinling', 'twins', 'beyond', 'wuyuetian', 'luodayou', 'fangdatong', 'taozhe', 'lizongsheng', 'mowenwei', 'fangdatong', 'wangsulong', 'maobuyi', 'zhoujielun', 'suyoupeng', 'lironghao', 'liudehua', 'she', 'zhangshaohan', 'xuezhiqian'
    ]
file_path_prefix = f"data/{singer_list[-1]}/"
# file_path_prefix = f"data/liangjingru/"

In [12]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,318,1775750401,顽疾,顽疾,顽疾,2026-04-10,2026
1,102636799,001Qu4I30eVFYb,演员,NaN,薛之谦,5062,002J4UUk29y8BY,绅士,989994,003y8dsH2wBHlo,261,1433433600,演员,演员,绅士,2015-06-05,2015
2,104775877,003ouHMP12glVD,其实,《妈妈像花儿一样》电视剧插曲,薛之谦,5062,002J4UUk29y8BY,意外,443691,000QgFcm0v8WaF,242,1384099200,其实,其实,意外,2013-11-11,2013
3,272125057,0013WPvt4fQH2b,天外来物,NaN,薛之谦,5062,002J4UUk29y8BY,天外来物,16596032,000K9Zp13TZp5s,257,1609344000,天外来物,天外来物,天外来物,2020-12-31,2020
4,233704383,002zfxmN2e1vLQ,陪你去流浪,NaN,薛之谦,5062,002J4UUk29y8BY,尘,7064087,000DMpJ73yeITP,274,1577376000,陪你去流浪,陪你去流浪,尘,2019-12-27,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,647958,002teWyP3uBNgN,马戏小丑,NaN,薛之谦,5062,002J4UUk29y8BY,未完成的歌,55085,0036fT613sAeZn,303,1259596800,马戏小丑,马戏小丑,未完成的歌,2009-12-01,2009
111,650356745,004W4x5G4Kjpen,粉钻 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,250,1773936000,粉钻,粉钻,NaN,2026-03-20,2026
112,1333806,000fjX1m12reIT,快乐帮,NaN,薛之谦,5062,002J4UUk29y8BY,薛之谦,51504,003mUYW22JXKVK,243,1136044800,快乐帮,快乐帮,薛之谦,2006-01-01,2006
113,649823961,003Rpde21bLvWc,造物 (2026万兽之王世界巡回演唱会彩排版),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,64,1773763200,造物,造物,NaN,2026-03-18,2026


In [ ]:
# 五月天需要使用word_to_fix
# if file_path_prefix == "data/mayday/":
#     df_word = lyric_words_process(file_path_prefix, word_to_fix)
# else:
#     df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [13]:
# 词性解析
# hanlp暂时不需要word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
df_word

顽疾
粉钻 (2026万兽之王巡回演唱会广州站现场)
造物 (2026万兽之王世界巡回演唱会彩排版)
平庸 (2026万兽之王巡回演唱会广州站现场)


,song_id,word,pos,freq
0,201697348,的,u,17
1,201697348,感情,n,8
2,201697348,现在,t,6
3,201697348,都,d,6
4,201697348,你,r,6
...,...,...,...,...
11969,650193615,浪漫,a,1
11970,650193615,是,v,1
11971,650193615,别来无恙,l,1
11972,650193615,为,p,1


In [39]:
df_word[df_word['word'] == '就']

,song_id,word,pos,freq
35,201697348,就,d,2
255,102636799,就,d,3
389,214182478,就,d,6
528,1333803,就,d,3
584,107439472,就,d,3
...,...,...,...,...
11188,1257199,就,d,2
11240,1333806,就,d,5
11707,655167781,就,d,1
11735,650356745,就,d,2


In [14]:
# 重新读取
df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")
df_word_read

,song_id,word,pos,freq
0,201697348,的,u,17
1,201697348,感情,n,8
2,201697348,现在,t,6
3,201697348,都,d,6
4,201697348,你,r,6
...,...,...,...,...
11969,650193615,浪漫,a,1
11970,650193615,是,v,1
11971,650193615,别来无恙,l,1
11972,650193615,为,p,1


In [28]:
df_merged = words_data_merge(df_word_read, df_songs)
df_merged = df_merged.dropna(subset='song_name')
df_merged

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,...,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,word,pos,freq,is_chinese,is_english
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,的,u,15,1,0
1,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,我,r,12,1,0
2,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,你,r,7,1,0
3,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,越,d,6,1,0
4,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,不,d,5,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11969,650193615,002MQVWb3ShQhA,平庸 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,...,平庸,平庸,NaN,2026-03-20,2026,浪漫,a,1,1,0
11970,650193615,002MQVWb3ShQhA,平庸 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,...,平庸,平庸,NaN,2026-03-20,2026,是,v,1,1,0
11971,650193615,002MQVWb3ShQhA,平庸 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,...,平庸,平庸,NaN,2026-03-20,2026,别来无恙,l,1,1,0
11972,650193615,002MQVWb3ShQhA,平庸 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,...,平庸,平庸,NaN,2026-03-20,2026,为,p,1,1,0


In [29]:
df_merged[df_merged['pos'] == 'e']['word'].unique()

array(['wow', 'oh', '哎哟', '嘿', '啊啊', '噢', '哼哼', '哦', 'yeah'], dtype=object)

In [30]:
# 过滤中文词
df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
df_merged_chn
# 不过滤中文词
# df_merged_chn = df_merged.copy()
# 只保留中文和英文
# df_merged_chn = df_merged_chn[(df_merged_chn['is_chinese'] == 1) | (df_merged_chn['is_english'] == 1)]

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,...,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,word,pos,freq,is_chinese,is_english
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,的,u,15,1,0
1,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,我,r,12,1,0
2,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,你,r,7,1,0
3,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,越,d,6,1,0
4,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,顽疾,89256479,0033ZoPK0kL53m,...,顽疾,顽疾,顽疾,2026-04-10,2026,不,d,5,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11969,650193615,002MQVWb3ShQhA,平庸 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,...,平庸,平庸,NaN,2026-03-20,2026,浪漫,a,1,1,0
11970,650193615,002MQVWb3ShQhA,平庸 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,...,平庸,平庸,NaN,2026-03-20,2026,是,v,1,1,0
11971,650193615,002MQVWb3ShQhA,平庸 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,...,平庸,平庸,NaN,2026-03-20,2026,别来无恙,l,1,1,0
11972,650193615,002MQVWb3ShQhA,平庸 (2026万兽之王巡回演唱会广州站现场),NaN,薛之谦,0,0032fmHO2UDnV3,NaN,0,NaN,...,平庸,平庸,NaN,2026-03-20,2026,为,p,1,1,0


In [31]:
# 查看歌曲数
df_merged_chn['song_name_pure'].nunique()

115

In [32]:
# 虚拟专辑数据
df_songs_part = df_merged_chn[[
    'song_name_pure'
]].drop_duplicates(keep='first').reset_index(drop=True)
# 只保留120个
df_songs_part = df_songs_part.head(100)
df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                         1).astype(str)
df_songs_part['album_order'] = df_songs_part.index // 10
df_songs_part

,song_name_pure,album_name,album_order
0,顽疾,PART 1,0
1,演员,PART 1,0
2,其实,PART 1,0
3,天外来物,PART 1,0
4,陪你去流浪,PART 1,0
...,...,...,...
95,给我的爱人,PART 10,9
96,红尘女子,PART 10,9
97,memory,PART 10,9
98,爱情宣判,PART 10,9


In [33]:
# 虚拟专辑数据，index//12+1作为虚拟专辑
df_merged_chn = df_merged_chn.copy()
df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
df_merged_chn = df_merged_chn.drop(columns=['album_name'])
df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
# 删除album_order为空的数据
df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
df_merged_chn

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_id,album_mid,duration,...,publish_date,publish_year,word,pos,freq,is_chinese,is_english,album_name_raw,album_name,album_order
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,89256479,0033ZoPK0kL53m,318,...,2026-04-10,2026,的,u,15,1,0,顽疾,PART 1,0.0
1,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,89256479,0033ZoPK0kL53m,318,...,2026-04-10,2026,我,r,12,1,0,顽疾,PART 1,0.0
2,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,89256479,0033ZoPK0kL53m,318,...,2026-04-10,2026,你,r,7,1,0,顽疾,PART 1,0.0
3,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,89256479,0033ZoPK0kL53m,318,...,2026-04-10,2026,越,d,6,1,0,顽疾,PART 1,0.0
4,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,89256479,0033ZoPK0kL53m,318,...,2026-04-10,2026,不,d,5,1,0,顽疾,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10447,1771030,003ny8493uibDE,为什么,NaN,薛之谦,5062,002J4UUk29y8BY,122328,003qp8IX21MXH0,257,...,2012-08-15,2012,却,d,1,1,0,几个薛之谦,PART 10,9.0
10448,1771030,003ny8493uibDE,为什么,NaN,薛之谦,5062,002J4UUk29y8BY,122328,003qp8IX21MXH0,257,...,2012-08-15,2012,最后,f,1,1,0,几个薛之谦,PART 10,9.0
10449,1771030,003ny8493uibDE,为什么,NaN,薛之谦,5062,002J4UUk29y8BY,122328,003qp8IX21MXH0,257,...,2012-08-15,2012,发现,v,1,1,0,几个薛之谦,PART 10,9.0
10450,1771030,003ny8493uibDE,为什么,NaN,薛之谦,5062,002J4UUk29y8BY,122328,003qp8IX21MXH0,257,...,2012-08-15,2012,开心,a,1,1,0,几个薛之谦,PART 10,9.0


In [34]:
df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

In [35]:
# 数据查验
songs_n = df_merged_chn[df_merged_chn['pos'] == 'n']['song_name_pure'].unique().tolist()
songs_all = df_merged_chn['song_name_pure'].unique().tolist()
for i in songs_all:
    if i not in songs_n:
        print(i)

# 歌曲数据更新

In [36]:
df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)

df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,is_english,album_name_raw,album_name,album_order
0,655167781,000wnC614MLM2X,顽疾,NaN,薛之谦,5062,002J4UUk29y8BY,89256479,0033ZoPK0kL53m,318,1775750401,顽疾,顽疾,顽疾,2026-04-10,2026,0,顽疾,PART 1,0.0
1,102636799,001Qu4I30eVFYb,演员,NaN,薛之谦,5062,002J4UUk29y8BY,989994,003y8dsH2wBHlo,261,1433433600,演员,演员,绅士,2015-06-05,2015,0,绅士,PART 1,0.0
2,104775877,003ouHMP12glVD,其实,《妈妈像花儿一样》电视剧插曲,薛之谦,5062,002J4UUk29y8BY,443691,000QgFcm0v8WaF,242,1384099200,其实,其实,意外,2013-11-11,2013,0,意外,PART 1,0.0
3,272125057,0013WPvt4fQH2b,天外来物,NaN,薛之谦,5062,002J4UUk29y8BY,16596032,000K9Zp13TZp5s,257,1609344000,天外来物,天外来物,天外来物,2020-12-31,2020,0,天外来物,PART 1,0.0
4,233704383,002zfxmN2e1vLQ,陪你去流浪,NaN,薛之谦,5062,002J4UUk29y8BY,7064087,000DMpJ73yeITP,274,1577376000,陪你去流浪,陪你去流浪,尘,2019-12-27,2019,0,尘,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1257200,003khWub46zVRU,给我的爱人,NaN,薛之谦,5062,002J4UUk29y8BY,37055,001OdsAb1vBRlV,284,1227456000,给我的爱人,给我的爱人,深深爱过你,2008-11-24,2008,0,深深爱过你,PART 10,9.0
96,1333804,000Tn6zj2lK4ax,红尘女子,NaN,薛之谦,5062,002J4UUk29y8BY,51504,003mUYW22JXKVK,285,1136044800,红尘女子,红尘女子,薛之谦,2006-01-01,2006,0,薛之谦,PART 10,9.0
97,1333810,002yWllS3yY6yd,Memory,NaN,薛之谦,5062,002J4UUk29y8BY,51504,003mUYW22JXKVK,298,1136044800,Memory,memory,薛之谦,2006-01-01,2006,0,薛之谦,PART 10,9.0
98,1308681,003Drqy31LPuT1,爱情宣判,NaN,薛之谦,5062,002J4UUk29y8BY,32506,004PY3HK4cv5z4,274,1185811200,爱情宣判,爱情宣判,你过得好吗,2007-07-31,2007,0,你过得好吗,PART 10,9.0


In [37]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# 测试

In [ ]:
1260/500

In [ ]:
2318/2.52

In [ ]:
from datetime import datetime
datetime.now().month